## Changes: safer tokenization and error inspection

- Added a `clean_text` function that replaces URLs, phone numbers and monetary amounts with `<url>`, `<phone>`, and `<mon>` tokens and lowercases text.
- Added explicit `<PAD>` (index 0) and `<UNK>` (index 1) tokens in the vocabulary. Unknown words map to `<UNK>` at inference time.
- Updated the dataset to pad with the PAD index and to return cleaned text for inspection.
- Evaluation now collects misclassified examples and prints heuristic explanations (presence of special tokens, high OOV ratio, short/ambiguous messages, etc.).

These small changes make inference safer for unseen tokens and help diagnose model failures.

In [15]:
import pandas as pd
import numpy
import re 
import string
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [16]:
df = pd.read_csv('spam.csv',encoding='latin-1')
df['label']=df['label'].map({'ham':0,'spam':1})

# --- Preprocessing: replace URLs, phones, monetary amounts with special tokens ---
url_re = r'https?://\S+|www\.\S+'
phone_re = r'\b(?:\+?\d{1,3}[\s-]?)?(?:\(\d{2,3}\)|\d{2,3})[\s-]?\d{3,4}[\s-]?\d{3,4}\b'
money_re = r'\$\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*\s?(?:usd|dollars|eur|£|€)\b'

def clean_text(text):
    # lower-case
    text = str(text).lower()
    # replace urls, phones, money with special tokens (lowercase tokens)
    text = re.sub(url_re, '<url>', text)
    text = re.sub(phone_re, '<phone>', text)
    text = re.sub(money_re, '<mon>', text)
    # remove punctuation (but keep our special tokens angle brackets and letters/numbers)
    # replace punctuation except angle brackets used in tokens
    text = re.sub(r'[^a-z0-9<>\s]', ' ', text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
df['text_clean'] = df['text'].apply(clean_text)
df.head()

,label,text,text_clean
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...


In [17]:
all_words = []
# Build vocab from cleaned text
for text in df['text_clean']:
    all_words.extend(text.split())

words_count = Counter(all_words)
vocab_list = sorted(words_count, key=words_count.get, reverse=True)

# Reserve 0 for PAD, 1 for UNK
vocab = {'<PAD>': 0, '<UNK>': 1}
# start indexing from 2
for idx, word in enumerate(vocab_list, start=2):
    vocab[word] = idx

print(f"vocab size (including PAD/UNK): {len(vocab)}")
print('example index for <UNK>:', vocab.get('<UNK>'))

vocab size (including PAD/UNK): 8335
example index for <UNK>: 1


In [18]:
def tokenize(text, vocab):
    # map words to indices, unknown words -> <UNK> index
    unk_idx = vocab.get('<UNK>', 1)
    return [vocab.get(word, unk_idx) for word in str(text).split()]

# Encode using cleaned text
df['encoded_text'] = df['text_clean'].apply(lambda x: tokenize(x, vocab))

print('Original text:', df['text'][0])
print('Cleaned text:', df['text_clean'][0])
print('Encoded text:', df['encoded_text'][0])

Original text: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Cleaned text: go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat
Encoded text: [55, 464, 4188, 818, 735, 646, 72, 9, 1282, 95, 129, 335, 1283, 155, 2867, 1284, 64, 63, 4189, 140]


In [19]:
class SpamDataset(Dataset):
    def __init__(self, data, max_len=20):
        self.data = data.reset_index(drop=True)
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        row = self.data.iloc[index]
        label = row['label']
        encoded_text = row['encoded_text']
        cleaned = row.get('text_clean', row.get('text', ''))
        
        # Padding/Truncating to ensure fixed length, PAD index is 0
        pad_idx = 0
        if len(encoded_text) < self.max_len:
            padded = encoded_text + [pad_idx] * (self.max_len - len(encoded_text))
        else:
            padded = encoded_text[:self.max_len]
            
        return torch.tensor(padded, dtype=torch.long), torch.tensor(label, dtype=torch.float), cleaned


train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])

train_dataset = SpamDataset(train_df)
test_dataset = SpamDataset(test_df)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 3900
Number of testing samples: 1672


In [20]:
class SpamClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size=50):
        super(SpamClassifier, self).__init__()

        # vocab already includes PAD and UNK; embedding num_embeddings must be vocab_size (largest index + 1)
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=0)

        self.fc = nn.Linear(embed_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        
        sentence_embedded = embedded.mean(dim=1)

        out = self.fc(sentence_embedded)
        out = self.sigmoid(out)
        return out

# vocab dict maps tokens to indices; embedding num_embeddings should be max_index+1
max_index = max(vocab.values())
model = SpamClassifier(vocab_size=(max_index+1), embed_size=50).to(device)
print(model)

# Num of parms
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")

SpamClassifier(
  (embedding): Embedding(8335, 50, padding_idx=0)
  (fc): Linear(in_features=50, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Total trainable parameters: 416801


In [21]:
learning_rate = 0.001
epochs = 10

criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for texts, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        texts, labels = texts.to(device), labels.to(device).float().unsqueeze(1)

        optimiser.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1/10: 100%|██████████| 61/61 [00:00<00:00, 157.83it/s]


Epoch 1, Loss: 0.6249


Epoch 2/10: 100%|██████████| 61/61 [00:00<00:00, 176.55it/s]


Epoch 2, Loss: 0.5411


Epoch 3/10: 100%|██████████| 61/61 [00:00<00:00, 180.19it/s]


...

Epoch 8, Loss: 0.1964


Epoch 9/10: 100%|██████████| 61/61 [00:00<00:00, 184.00it/s]


Epoch 9, Loss: 0.1701


Epoch 10/10: 100%|██████████| 61/61 [00:00<00:00, 184.80it/s]

Epoch 10, Loss: 0.1493


In [22]:
model.eval()
predictions = []
actuals = []
misclassified = []

with torch.no_grad():
    for texts, labels, cleaned_texts in test_loader:
        texts, labels = texts.to(device), labels.to(device).float().unsqueeze(1)
        outputs = model(texts)
        probs = outputs.squeeze(1)
        preds = (probs >= 0.5).float()
        
        predictions.extend(preds.cpu().numpy())
        actuals.extend(labels.cpu().numpy())
        
        # store misclassified examples for inspection
        for i in range(len(preds)):
            if preds[i].item() != labels[i].item():
                prob = probs[i].item()
                true_label = int(labels[i].item())
                pred_label = int(preds[i].item())
                cleaned = cleaned_texts[i]
                misclassified.append({
                    'text_clean': cleaned,
                    'true': true_label,
                    'pred': pred_label,
                    'prob': prob
                })

accuracy = accuracy_score(actuals, predictions)
print(f"Test Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(actuals, predictions, target_names=['ham', 'spam']))

# Heuristic inspection of misclassified examples
print(f"Total misclassified: {len(misclassified)}\n")

def explain_misclassified(example, vocab):
    text = example['text_clean']
    words = text.split()
    reasons = []
    # check for special tokens (lowercase since we cleaned to lowercase)
    if '<url>' in text:
        reasons.append('contains <URL> token')
    if '<phone>' in text:
        reasons.append('contains <PHONE> token')
    if '<mon>' in text:
        reasons.append('contains <MON> token')
    # many unknowns -> might be OOV heavy
    unk_idx = vocab.get('<UNK>', 1)
    unk_count = sum(1 for w in words if vocab.get(w, unk_idx) == unk_idx)
    if len(words) > 0 and (unk_count / len(words)) > 0.4:
        reasons.append(f'high OOV ratio ({unk_count}/{len(words)})')
    # short ambiguous messages
    if len(words) <= 3:
        reasons.append('very short/ambiguous message')
    # fallback
    if not reasons:
        reasons.append('ambiguous or subtle semantic cues; model uses averaged embeddings so word order and rare cues lost')
    return reasons

# print a handful of misclassified examples with reasons
for i, ex in enumerate(misclassified[:20]):
    reasons = explain_misclassified(ex, vocab)
    print(f"Example {i+1}: True={ex['true']}, Pred={ex['pred']}, Prob={ex['prob']:.3f}")
    print(f"Cleaned: {ex['text_clean']}")
    print('Likely causes:', '; '.join(reasons))
    print('-' * 60)

# Summary counts of common reasons
from collections import Counter
all_reasons = []
for ex in misclassified:
    all_reasons.extend(explain_misclassified(ex, vocab))
print('\nTop failure reasons:')
for r, c in Counter(all_reasons).most_common(10):
    print(f"{r}: {c}")

Test Accuracy: 0.9737
Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98      1448
        spam       0.97      0.83      0.89       224

    accuracy                           0.97      1672
   macro avg       0.97      0.91      0.94      1672
weighted avg       0.97      0.97      0.97      1672

Total misclassified: 44

Example 1: True=1, Pred=0, Prob=0.480
Cleaned: themob>hit the link to get a premium pink panther game the new no 1 from sugababes a crazy zebra animation or a badass hoody wallpaper all 4 free
Likely causes: ambiguous or subtle semantic cues; model uses averaged embeddings so word order and rare cues lost
------------------------------------------------------------
Example 2: True=1, Pred=0, Prob=0.264
Cleaned: more people are dogging in your area now call <phone> and join like minded guys why not arrange 1 yourself there s 1 this evening a 1 50 minapn ls278bb
Likely causes: contains <PHONE> toke